In [599]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import pyspark
from pyspark.sql.functions import col
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, regexp_replace, to_date, count, sum as spark_sum, isnan
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

In [600]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

# 1. Clickstream Dataset

In [ ]:
clickstream_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv("./datamart/bronze/feature_clickstream/bronze_clickstream_*.csv")
)
clickstream_df.printSchema()
print("Row count:", clickstream_df.count())

In [ ]:
clickstream_df.show(5)

In [ ]:
clickstream_df.describe()

In [ ]:
# check null column
clickstream_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in clickstream_df.columns
]).show()

In [ ]:
# how many rows per snapshot month?
clickstream_df.groupBy("snapshot_date").count().orderBy("snapshot_date").show(30)